# Cell Type Importance via Attention Weights
### Which cell types drive Normal vs Crohn disease classification?

**Approach:** Extract per-cell softmax attention weights from your trained
`AttentionPoolingModel`. These weights are produced during every forward pass —
we were discarding them with `_` during training. Here we capture them,
map each cell back to its cell type (from the original `.h5ad`), and aggregate.

**Pipeline:**
1. Load original `.h5ad` → build `donor → cell_type array` lookup
2. For each fold: load `best_model_fold{N}.pt` → run inference on **validation donors**
3. Capture attention weights `(n_cells,)` per donor
4. Aggregate: mean attention weight per `(donor × cell_type)`
5. **Fig A** — Ranked cell type importance boxplot
6. **Fig B** — Normal vs Crohn per top cell types + Mann-Whitney U stats
7. **Fig C** — Attention weight heatmap across all 5 folds (consistency check)
8. **Fig D** — Weight distribution per cell type (violin + strip, per donor)
9. **Fig E** — Per-donor attention weight profile (stacked bar, all val donors)


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader

from attention_model import (
    AttentionPoolingModel,
    PatientDataset,
    collate_fn,
    load_all_patients_flat,
    set_seed,
)

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")


## Configuration
Match exactly what you used in `train_5fold_cv.ipynb`.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────
H5AD_PATH       = "SCP1884_subset_50_normalised_expressions_with_embeddings_cp.h5ad"
ALL_DATA_DIR    = "data/all/"
ALL_LABELS_PATH = "data/all_labels.npy"
OUTPUT_DIR      = "cell_type_importance"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────
D_H          = 512          # scGPT embedding dim
NUM_CLASSES  = 2
BATCH_SIZE   = 8
N_FOLDS      = 5
RANDOM_SEED  = 42

# ── Data ──────────────────────────────────────────────────────────
DONOR_COL     = "donor_id"
CELL_TYPE_COL = "cell_type"   # obs column name — check adata.obs.columns

# ── Analysis ──────────────────────────────────────────────────────
MIN_DONORS_PER_CT = 3    # drop cell types seen in fewer donors
TOP_N_CTS         = 8   # top cell types for Fig B, D, E

LABEL_NAMES     = {0: "Normal", 1: "Crohn disease"}
DISEASE_PALETTE = {"Normal": "#1f77b4", "Crohn disease": "#e74c3c"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Device : {DEVICE}")
print(f"Output : {OUTPUT_DIR}/")


## Step 1 — Build donor → cell type lookup from `.h5ad`

Each donor's `.npy` file has rows in the **same order** as `adata[adata.obs.donor_id == donor]`
because `dataset_creation_kong.ipynb` iterates over `sorted(adata.obs[donor_col].unique())`
and slices `adata[donor_mask]` without shuffling.

We build a dict `donor_id → np.array of cell type strings` that mirrors this ordering.


In [ ]:
print(f"Loading {H5AD_PATH} ...")
adata = sc.read_h5ad(H5AD_PATH)

print(f"  Cells         : {adata.shape[0]:,}")
print(f"  obs columns   : {adata.obs.columns.tolist()}")
print(f"  Cell types    : {sorted(adata.obs[CELL_TYPE_COL].unique().tolist())}")
print(f"  Unique donors : {adata.obs[DONOR_COL].nunique()}")


In [ ]:
# Build lookup: donor_id (as used in .npy filenames) → cell type array
# dataset_creation uses: safe_did = str(did).replace("/", "_").replace(" ", "_")
# We must apply the same transform when looking up from adata

donor_celltype_lookup = {}

for raw_did in sorted(adata.obs[DONOR_COL].unique()):
    safe_did = str(raw_did).replace("/", "_").replace(" ", "_")
    mask = adata.obs[DONOR_COL] == raw_did
    cell_types = adata.obs.loc[mask, CELL_TYPE_COL].values.astype(str)
    donor_celltype_lookup[safe_did] = cell_types

print(f"Built lookup for {len(donor_celltype_lookup)} donors")

# Sanity check against .npy files
npy_donors = sorted([
    f.replace("_embeddings.npy", "")
    for f in os.listdir(ALL_DATA_DIR)
    if f.endswith("_embeddings.npy")
])

missing = [d for d in npy_donors if d not in donor_celltype_lookup]
if missing:
    print(f"WARNING — {len(missing)} .npy donors not found in adata: {missing}")
else:
    print(f"All {len(npy_donors)} .npy donors found in adata ✓")

# Verify row counts match between .npy and adata
print("\nRow count check (first 5 donors):")
for did in npy_donors[:5]:
    Z_shape   = np.load(os.path.join(ALL_DATA_DIR, f"{did}_embeddings.npy")).shape
    ct_len    = len(donor_celltype_lookup[did])
    match_sym = "✓" if Z_shape[0] == ct_len else "✗ MISMATCH"
    print(f"  {did:<35}  npy={Z_shape[0]:>4} cells  adata={ct_len:>4} cells  {match_sym}")


## Step 2 — Load patient embeddings (same as training)

In [ ]:
patient_list, donor_ids, all_labels = load_all_patients_flat(
    ALL_DATA_DIR, ALL_LABELS_PATH
)

print(f"Total donors : {len(patient_list)}")
print(f"  Normal (0) : {(all_labels == 0).sum()}")
print(f"  Crohn  (1) : {(all_labels == 1).sum()}")


## Step 3 — `AttentionCellTypeAnalyzer`

The attention weights `(n_cells,)` are already computed in every forward pass.
In training they are returned as the third element but never used.
Here we capture them and aggregate by cell type.


In [ ]:
class AttentionCellTypeAnalyzer:
    """
    Extracts per-cell softmax attention weights from a trained
    AttentionPoolingModel and aggregates them by cell type.

    Attention weights directly score each cell's contribution to the
    patient embedding used for classification — no additional computation
    needed beyond a standard forward pass.
    """

    def __init__(self, model_path, d_h=512, num_classes=2, device=torch.device("cpu")):
        self.device = device

        self.model = AttentionPoolingModel(
            d_h=d_h, num_classes=num_classes
        ).to(device)

        self.model.load_state_dict(
            torch.load(model_path, map_location=device)
        )
        self.model.eval()
        print(f"  Loaded: {model_path} ✓")

    @torch.no_grad()
    def get_cell_weights(self, patient_data):
        """
        Run one patient through the model and return per-cell attention weights.

        Args
        ----
        patient_data : dict with Z (n_cells × d_h), mask, label

        Returns
        -------
        weights : np.ndarray (n_cells,)  — softmax weights, sum to 1
        label   : int
        """
        Z    = patient_data["Z"].unsqueeze(0).to(self.device)   # (1, n_cells, d_h)
        mask = patient_data["mask"].unsqueeze(0).to(self.device)
        Z    = F.normalize(Z, dim=-1)   # same as training

        _, _, weights = self.model(Z, mask)     # weights: (1, n_cells)
        return weights.squeeze(0).cpu().numpy(), patient_data["label"]

    def aggregate_by_celltype(self, weights, cell_types):
        """
        Compute mean attention weight per cell type for one donor.

        Args
        ----
        weights    : np.ndarray (n_cells,)
        cell_types : np.ndarray (n_cells,)  str labels

        Returns
        -------
        dict {cell_type: mean_attention_weight}
        """
        result = {}
        for ct in np.unique(cell_types):
            mask = cell_types == ct
            result[ct] = float(weights[mask].mean())
        return result

    def full_weight_distribution(self, weights, cell_types):
        """
        Return the raw weight array split by cell type — used for violin plots
        and per-donor profile (Fig D, Fig E).

        Returns
        -------
        dict {cell_type: np.ndarray of individual cell weights}
        """
        result = {}
        for ct in np.unique(cell_types):
            mask = cell_types == ct
            result[ct] = weights[mask]
        return result


## Step 4 — Extract attention weights across all 5 folds

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# records_mean : one row per (donor, cell_type) — mean attention weight
# records_raw  : one row per cell — raw attention weight (for distributions)
records_mean = []
records_raw  = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(patient_list)), all_labels), start=1
):
    print(f"\n{'='*55}")
    print(f"  Fold {fold}  —  {len(val_idx)} validation donors")
    print(f"{'='*55}")

    analyzer = AttentionCellTypeAnalyzer(
        model_path=f"best_model_fold{fold}.pt",
        d_h=D_H, num_classes=NUM_CLASSES, device=DEVICE,
    )

    for idx in val_idx:
        did        = donor_ids[idx]
        true_label = int(all_labels[idx])
        disease    = LABEL_NAMES[true_label]

        if did not in donor_celltype_lookup:
            print(f"  WARNING: {did} not in lookup — skipping")
            continue

        cell_types = donor_celltype_lookup[did]
        weights, _ = analyzer.get_cell_weights(patient_list[idx])

        # Align lengths (DataLoader may not subsample, but guard anyway)
        n = min(len(weights), len(cell_types))
        weights, cell_types = weights[:n], cell_types[:n]

        # ── Mean per cell type ────────────────────────────────────
        ct_means = analyzer.aggregate_by_celltype(weights, cell_types)
        for ct, mean_w in ct_means.items():
            records_mean.append({
                "donor_id"       : did,
                "label"          : true_label,
                "disease"        : disease,
                "fold"           : fold,
                "cell_type"      : ct,
                "mean_attention" : mean_w,
            })

        # ── Raw weights per cell ──────────────────────────────────
        ct_raw = analyzer.full_weight_distribution(weights, cell_types)
        for ct, raw_w in ct_raw.items():
            for w in raw_w:
                records_raw.append({
                    "donor_id"  : did,
                    "label"     : true_label,
                    "disease"   : disease,
                    "fold"      : fold,
                    "cell_type" : ct,
                    "weight"    : float(w),
                })

    print(f"  Done ✓")

# ── Build DataFrames ──────────────────────────────────────────────
df_mean = pd.DataFrame(records_mean)
df_raw  = pd.DataFrame(records_raw)

print(f"\nMean-level records : {len(df_mean)}  (donors × cell types)")
print(f"Cell-level records  : {len(df_raw)}  (individual cells)")
print(f"Cell types found   : {sorted(df_mean['cell_type'].unique().tolist())}")

# Save
df_mean.to_csv(os.path.join(OUTPUT_DIR, "attention_mean_per_donor_celltype.csv"), index=False)
df_raw.to_csv(os.path.join(OUTPUT_DIR,  "attention_raw_per_cell.csv"),            index=False)
print(f"\nSaved CSVs to {OUTPUT_DIR}/ ✓")


## Step 5 — Filter and rank cell types

In [ ]:
# Drop cell types seen in too few donors
ct_donor_counts = df_mean.groupby("cell_type")["donor_id"].nunique()
common_cts      = ct_donor_counts[ct_donor_counts >= MIN_DONORS_PER_CT].index
df_plot         = df_mean[df_mean["cell_type"].isin(common_cts)].copy()

# Rank by median mean attention
ct_order = (
    df_plot.groupby("cell_type")["mean_attention"]
    .median().sort_values().index.tolist()
)
top_cts = ct_order[-TOP_N_CTS:]

print(f"Cell types with >= {MIN_DONORS_PER_CT} donors : {len(ct_order)}")
print(f"Top {TOP_N_CTS} by median attention:")
for ct in reversed(top_cts):
    med = df_plot[df_plot["cell_type"]==ct]["mean_attention"].median()
    n   = df_plot[df_plot["cell_type"]==ct]["donor_id"].nunique()
    print(f"  {ct:<45}  median={med:.5f}  n_donors={n}")


## Fig A — Ranked Cell Type Importance

All validation donors combined across 5 folds.
Each point = one donor's mean attention weight for that cell type.


In [ ]:
fig, ax = plt.subplots(figsize=(10, max(6, len(ct_order) * 0.5)))

rng = np.random.default_rng(42)
data_groups = [
    df_plot.loc[df_plot["cell_type"] == ct, "mean_attention"].values
    for ct in ct_order
]

bp = ax.boxplot(
    data_groups,
    vert=False,
    labels=ct_order,
    showmeans=True,
    patch_artist=True,
    boxprops=dict(facecolor="#d0e4f7", color="#2c5f8a"),
    medianprops=dict(color="#e74c3c", linewidth=2),
    meanprops=dict(marker="D", markerfacecolor="#e67e22",
                   markeredgecolor="white", markersize=6),
    whiskerprops=dict(color="#2c5f8a"),
    capprops=dict(color="#2c5f8a"),
)

for i, gd in enumerate(data_groups, start=1):
    jitter = rng.uniform(-0.18, 0.18, size=len(gd))
    ax.scatter(gd, np.full_like(gd, i, dtype=float) + jitter,
               alpha=0.5, color="steelblue", s=22, zorder=3)

ax.set_xlabel("Mean Attention Weight per Donor", fontsize=12)
ax.set_title(
    "Cell Type Importance — Normal vs Crohn Disease Classification\n"
    "(Softmax attention weights, 5-fold CV validation donors)",
    fontsize=13, fontweight="bold"
)
ax.tick_params(axis="y", labelsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, linestyle="--", alpha=0.3, axis="x")
plt.tight_layout()

path = os.path.join(OUTPUT_DIR, "fig_A_cell_type_importance.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")


## Fig B — Normal vs Crohn per Top Cell Types

Mann-Whitney U test per cell type.
Stars: * p<0.05 · ** p<0.01 · *** p<0.001


In [ ]:
df_top = df_plot[df_plot["cell_type"].isin(top_cts)].copy()
df_top["cell_type"] = pd.Categorical(df_top["cell_type"], categories=top_cts, ordered=True)

fig, ax = plt.subplots(figsize=(max(10, len(top_cts) * 1.5), 7))

sns.boxplot(
    data=df_top, x="cell_type", y="mean_attention",
    hue="disease", palette=DISEASE_PALETTE,
    dodge=True, width=0.6, ax=ax,
    hue_order=["Normal", "Crohn disease"],
    linewidth=1.2,
)
sns.stripplot(
    data=df_top, x="cell_type", y="mean_attention",
    hue="disease", palette=DISEASE_PALETTE,
    dodge=True, ax=ax, alpha=0.45, size=5, jitter=True,
    hue_order=["Normal", "Crohn disease"],
    legend=False,
)

# Mann-Whitney U p-values
y_max = df_top["mean_attention"].max()
for i, ct in enumerate(top_cts):
    g_n = df_top[(df_top["cell_type"]==ct) & (df_top["disease"]=="Normal")]["mean_attention"]
    g_c = df_top[(df_top["cell_type"]==ct) & (df_top["disease"]=="Crohn disease")]["mean_attention"]
    if len(g_n) < 2 or len(g_c) < 2:
        continue
    _, pval = mannwhitneyu(g_n, g_c, alternative="two-sided")
    stars = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    ax.text(i, y_max * 1.06, stars, ha="center", fontsize=13,
            color="#2c3e50", fontweight="bold")

ax.set_xlabel("Cell Type", fontsize=12)
ax.set_ylabel("Mean Attention Weight", fontsize=12)
ax.set_title(
    "Attention Weight: Normal vs Crohn Disease\n"
    "(* p<0.05  ** p<0.01  *** p<0.001,  Mann-Whitney U)",
    fontsize=13, fontweight="bold"
)
ax.tick_params(axis="x", rotation=30)

# Deduplicate legend
handles, labs = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labs):
    if l not in seen: seen[l] = h
ax.legend(seen.values(), seen.keys(), title="Disease", fontsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

path = os.path.join(OUTPUT_DIR, "fig_B_normal_vs_crohn.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")


## Fig C — Cross-fold Consistency Heatmap

Mean attention weight per (fold × cell type).
Consistent colours across folds = robust, reproducible signal.


In [ ]:
heatmap_data = (
    df_plot[df_plot["cell_type"].isin(top_cts)]
    .groupby(["fold", "cell_type"])["mean_attention"]
    .mean()
    .unstack("cell_type")
    .reindex(columns=top_cts)
)

fig, ax = plt.subplots(figsize=(max(10, len(top_cts) * 1.3), 4))

sns.heatmap(
    heatmap_data,
    annot=True, fmt=".4f",
    cmap="Blues",
    linewidths=0.5, linecolor="white",
    ax=ax,
    cbar_kws={"label": "Mean attention weight"},
)

ax.set_xlabel("Cell Type", fontsize=12)
ax.set_ylabel("Fold", fontsize=12)
ax.set_title(
    "Mean Attention Weight per Cell Type × Fold\n"
    "(similar pattern across folds = robust signal)",
    fontsize=12, fontweight="bold"
)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()

path = os.path.join(OUTPUT_DIR, "fig_C_fold_consistency_heatmap.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")


## Fig D — Weight Distribution per Cell Type (Violin + Strip)

Uses **raw per-cell weights** rather than per-donor means.
This shows the spread of attention within each cell type across all validation cells,
split by disease — revealing whether individual cells of that type receive
consistently high or variable attention.


In [ ]:
df_raw_top = df_raw[df_raw["cell_type"].isin(top_cts)].copy()
df_raw_top["cell_type"] = pd.Categorical(
    df_raw_top["cell_type"], categories=top_cts, ordered=True
)

fig, ax = plt.subplots(figsize=(max(12, len(top_cts) * 1.8), 7))

sns.violinplot(
    data=df_raw_top, x="cell_type", y="weight",
    hue="disease", palette=DISEASE_PALETTE,
    dodge=True, inner=None, alpha=0.55, linewidth=1,
    hue_order=["Normal", "Crohn disease"],
    ax=ax,
)
sns.stripplot(
    data=df_raw_top.sample(min(len(df_raw_top), 3000), random_state=42),
    x="cell_type", y="weight",
    hue="disease", palette=DISEASE_PALETTE,
    dodge=True, alpha=0.25, size=2.5, jitter=True,
    hue_order=["Normal", "Crohn disease"],
    ax=ax, legend=False,
)

ax.set_xlabel("Cell Type", fontsize=12)
ax.set_ylabel("Per-cell Attention Weight", fontsize=12)
ax.set_title(
    "Per-cell Attention Weight Distribution by Cell Type\n"
    "(raw softmax weights — each point = one cell)",
    fontsize=13, fontweight="bold"
)
ax.tick_params(axis="x", rotation=30)

handles, labs = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labs):
    if l not in seen: seen[l] = h
ax.legend(seen.values(), seen.keys(), title="Disease", fontsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

path = os.path.join(OUTPUT_DIR, "fig_D_weight_distribution_violin.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")


## Fig E — Per-donor Attention Profile (Stacked Bar)

Each bar = one validation donor. Bar segments = fraction of total attention
assigned to each cell type. Donors sorted by disease then by dominant cell type.
This answers: *"does every Crohn donor show the same cell type dominance?"*


In [ ]:
# Pivot to wide: rows=donors, cols=cell_types, values=mean_attention
df_pivot = (
    df_mean[df_mean["cell_type"].isin(top_cts)]
    .pivot_table(index=["donor_id", "disease"], columns="cell_type",
                 values="mean_attention", aggfunc="mean")
    .reindex(columns=top_cts)
    .fillna(0)
)

# Normalise each row to sum to 1 (fraction of attention within top CTs)
df_norm = df_pivot.div(df_pivot.sum(axis=1), axis=0)

# Sort: disease first, then by top cell type descending
df_norm = df_norm.reset_index()
df_norm = df_norm.sort_values(
    ["disease"] + [top_cts[-1]],
    ascending=[True, False]
)

donor_labels_plot = df_norm["donor_id"].values
disease_labels    = df_norm["disease"].values
plot_data         = df_norm[top_cts].values

palette = sns.color_palette("tab10", n_colors=len(top_cts))
ct_colors = {ct: palette[i] for i, ct in enumerate(top_cts)}

fig, ax = plt.subplots(figsize=(max(14, len(donor_labels_plot) * 0.5), 6))

bottoms = np.zeros(len(donor_labels_plot))
for j, ct in enumerate(top_cts):
    vals = plot_data[:, j]
    bars = ax.bar(
        range(len(donor_labels_plot)), vals, bottom=bottoms,
        label=ct, color=ct_colors[ct], edgecolor="none", alpha=0.9
    )
    bottoms += vals

# Disease boundary line
disease_change = np.where(np.diff(disease_labels != disease_labels[0]))[0]
if len(disease_change) > 0:
    ax.axvline(disease_change[0] + 0.5, color="black", linewidth=2,
               linestyle="--", alpha=0.7, label="Disease boundary")

# Colour x-tick labels by disease
ax.set_xticks(range(len(donor_labels_plot)))
ax.set_xticklabels(donor_labels_plot, rotation=45, ha="right", fontsize=7)
for tick, dis in zip(ax.get_xticklabels(), disease_labels):
    tick.set_color(DISEASE_PALETTE[dis])

ax.set_ylabel("Fraction of Attention (top cell types)", fontsize=12)
ax.set_title(
    "Per-donor Attention Profile across Top Cell Types\n"
    "(red labels = Crohn disease, blue = Normal)",
    fontsize=13, fontweight="bold"
)
ax.set_ylim(0, 1.05)
ax.legend(bbox_to_anchor=(1.01, 0.5), loc="center left", fontsize=9, title="Cell Type")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

path = os.path.join(OUTPUT_DIR, "fig_E_per_donor_attention_profile.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")


## Summary Table — Cell Type Statistics

In [ ]:
summary = (
    df_plot
    .groupby(["cell_type", "disease"])["mean_attention"]
    .agg(n_donors="count", median="median", mean="mean", std="std")
    .reset_index()
    .sort_values("median", ascending=False)
)

print(summary.to_string(index=False))
summary.to_csv(os.path.join(OUTPUT_DIR, "cell_type_summary.csv"), index=False)
print(f"\nSaved → {OUTPUT_DIR}/cell_type_summary.csv")


In [ ]:
print("\nAll outputs:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size  = os.path.getsize(fpath)
    print(f"  {f:<50} {size/1024:>7.1f} KB")
